# 2 · Search WorldPop with browser Python

This tutorial uses the same HTTPS STAC API as the map. The async helper uses `pyodide.http.pyfetch` inside Pyodide. In normal Python it uses a standard-library adapter for local tests. Browser CORS still applies. Run the cells in order; if the provider is unavailable, rerun the connection cell later. No credentials are needed.

In [ ]:
from stac_browser import STACBrowser, PRESETS
client = await STACBrowser(PRESETS['worldpop']).connect()
print(client.root.get('title', client.root['id']))
print(len(client.collections), 'collections loaded')
[(c['id'], c.get('title', c['id'])) for c in client.collections[:8]]

## A bounded query

This rectangle covers Uganda and nearby areas. `bbox` is **west, south, east, north**, in WGS84 longitude/latitude. The collection ID is a country ISO3 code; WorldPop datetime can be a release date: inspect properties.year for the population reference year. The page limit is 5. Remove dates or broaden the bbox when no items match.

In [ ]:
items = await client.search(bbox=[29, -2, 35, 5], collection='UGA', limit=5)
print(len(items), 'items loaded')
for item in items:
    print(item['id'], item['properties'].get('year'), item['properties'].get('project'), item['properties'].get('datetime'), list(item.get('assets', {})))

## Preserve attribution and provenance

Items retain their original properties, links, and assets. Consult the collection's license and providers before downloading or publishing any data. This project’s MIT software license does not replace dataset licenses.

In [ ]:
collection = next((c for c in client.collections if c['id'] == 'UGA'), {})
print('License:', collection.get('license', 'consult provider'))
print('Providers:', collection.get('providers', []))
if items:
    client.save('worldpop')
    print('Saved worldpop-items.geojson and worldpop-search.json. Download them from the file browser.')
else:
    print('No data exported: broaden the query and retry.')
client.provenance()

## Load another page deliberately

Call `await client.more()` to load exactly one next page. It follows advertised GET/POST pagination, deduplicates collection/item IDs, and caps results at 500. It never walks every page automatically. To download your results, use the file browser’s Download command; browser storage can be cleared.